# CircuitSight — Fine-Tuning & Evaluation (Qwen2.5-VL-3B, QLoRA)

Trains a small vision-language model to **read a circuit schematic and solve it** (components → topology → equations → values), using the dataset produced by `CircuitSight_dataset_generation.ipynb`.

**Run order:**
1. Setup + GPU check
2. Load the dataset (from Drive or an uploaded zip)
3. Load Qwen2.5-VL-3B (4-bit) + LoRA
4. **Day-1 OOM smoke test — the go/no-go gate.** Run this *before* committing to a full run. If it OOMs, follow the fallback notes (lower resolution or smaller model) before proceeding.
5. Full training
6. Inference
7. **Evaluation harness** — scores base vs. tuned on: component accuracy, R_eq accuracy, final-answer accuracy, and — separately — **fabrication rate** vs. **honest-abstention rate**.

Needs a GPU runtime (Runtime → Change runtime type → GPU). With Colab credits, an **A100 40GB** is comfortable for the 3B; a T4/L4 works at lower resolution or with SmolVLM.


## 1. Setup

In [ ]:
# Install a torch version Unsloth supports (let pip choose the right CUDA build).
!pip install -q "torch==2.6.0" "torchvision==0.21.0"
!pip install -q unsloth sympy
print("installed — RESTART SESSION before importing")

In [ ]:
import torch; print("torch", torch.__version__)
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
print("unsloth ready")

torch 2.10.0+cu128
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth ready


## 2. Config

In [ ]:
CFG = dict(
    MODEL       = "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit",  # fallback: SmolVLM / a 2B if VRAM is tight
    MAX_IMAGE_PX= 512,     # longest image side; the main VRAM + speed knob. 512 ~= 2x faster than 768 on a T4.
    LORA_R      = 16, LORA_ALPHA = 16,
    BATCH       = 2, GRAD_ACCUM = 4,   # effective batch = BATCH*GRAD_ACCUM = 8. Bigger BATCH = faster but more VRAM; run the §4 smoke test first. OOM? -> BATCH=1, GRAD_ACCUM=8 (same effective 8, no extra VRAM). A100: BATCH 4-8 ok.
    MAX_STEPS   = 350,     # <-- caps optimizer steps (~30-40 min on a T4). Set to None to train a full epoch.
    EPOCHS      = 1,       # only used when MAX_STEPS is None
    LR          = 2e-4,
    MAX_LEN     = 2048,
    SEED        = 3407,
    DATA_DIR    = "circuitsight_dataset",   # unzipped dataset folder
    SAVE_TO_DRIVE = False,   # if True, snapshot the trained adapter to DRIVE_DIR/models (keep 2 most recent)
    DRIVE_DIR   = "/content/drive/MyDrive/CircuitSight",   # your project folder in Google Drive
    OUT_DIR     = "circuitsight_qlora",
)
# Why this many steps? EPOCHS=1 over ~10k images at effective batch BATCH*GRAD_ACCUM=4 is ~2,600
# optimizer steps (~6h on a T4) -- that is steps, not epochs. This narrow, structured behavior is
# learned in a few hundred steps, so we cap with MAX_STEPS. 350 steps ~= 2,800 images seen (effective batch 8); the
# trainer shuffles the full dataset, so all families + value-modes (numeric/symbolic/mixed) are
# sampled in proportion to the dataset mix (see CONFIG fractions). Watch the section-9 eval curve;
# raise MAX_STEPS (or set None) only if it's still climbing.
INSTRUCTION = ("You are a circuit analysis tutor. Look at the schematic and answer the question. "
    "First list every component, each with the image region it occupies as "
    "<box>[x0,y0,x1,y1]</box> in 0-1000 normalized coordinates. Then state the concepts used and "
    "the topology (what is in series/parallel). If there is a capacitor or inductor, apply its "
    "steady-state / t=0 behavior (a capacitor is open at steady state and a wire at t=0; an "
    "inductor is the reverse). Component values may be numbers (e.g. 100Ω, 12V) or symbols (e.g. "
    "R1, R2, V) — if they are symbols, give the answer as an algebraic expression in those symbols. "
    "Solve step by step, showing intermediate values (e.g. R_eq and each branch current), and "
    "end with a self-check. If a component value is not legible, say so and report the answer as "
    "null instead of guessing. End with a single line "
    "'FINAL: {\"quantity\":..., \"target_id\":..., \"value\":..., \"unit\":..., \"abstain\":false}'.")
CFG

## 3. Load the dataset

Get the dataset next to this notebook. Either mount Drive and point `DATA_DIR` at the unzipped folder, or upload the zip produced by the data-gen notebook.

In [ ]:
import os, json, zipfile

# Path to the FRESHLY regenerated dataset zip (numeric + symbolic + mixed, current schema).
# Regenerate with the dataset-generation notebook, download circuitsight_dataset.zip, upload it here.
# Do NOT reuse circuitsight_dataset_11k_4_3.zip — it predates the current schema/families (no symbolic
# values, no gold_answer.symbolic flag) and the eval will mismatch.
ZIP_PATH = "/content/circuitsight_dataset.zip"

# Unzip it. The dataset was zipped from INSIDE the dataset folder, so its contents
# (images/, train.jsonl, ...) sit at the zip root -> extract into CFG["DATA_DIR"].
assert os.path.exists(ZIP_PATH), f"{ZIP_PATH} not found - upload the fresh zip there first (folder icon, left sidebar)."
os.makedirs(CFG["DATA_DIR"], exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(CFG["DATA_DIR"])
print("unzipped into", CFG["DATA_DIR"])

IMG_DIR = os.path.join(CFG["DATA_DIR"], "images")
assert os.path.isdir(IMG_DIR), f"expected {IMG_DIR} after unzip - check the zip's internal structure"

def load_jsonl(name):
    p = os.path.join(CFG["DATA_DIR"], name)
    return [json.loads(l) for l in open(p)] if os.path.exists(p) else []

train_rows = load_jsonl("train.jsonl")
val_rows   = load_jsonl("val_synthetic.jsonl")
print(f"train: {len(train_rows)}  val: {len(val_rows)}")
print("example keys:", list(train_rows[0].keys()))
# sanity: confirm this is the FRESH dataset (has value_mode incl. symbolic/mixed)
from collections import Counter
print("value_mode mix:", Counter(r.get("value_mode","MISSING->stale zip!") for r in train_rows))

# --- alternatives to upload the zip ---
# from google.colab import files; up = files.upload(); ZIP_PATH = "/content/"+next(iter(up))
# from google.colab import drive; drive.mount('/content/drive')
# !cp /content/drive/MyDrive/CircuitSight/circuitsight_dataset.zip /content/

## 4. Format as vision chat samples

Each row → a user turn (image + instruction + question) and an assistant turn (the worked solution). Images are loaded as RGB PIL and downsized to `MAX_IMAGE_PX` — passing real PIL images (not paths) avoids the common *'could not make a flat list of images'* collator error.

In [ ]:
# All imports the training + eval cells rely on (safe to re-run anytime).
import os, json, zipfile, re, random, time
import torch
import numpy as np
from PIL import Image, ImageDraw
from datasets import Dataset
from collections import Counter

# unsloth / trl (already installed; just binding the names in this session)
from unsloth import FastVisionModel, is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

print("all imports loaded — PIL, torch, datasets, unsloth, trl ready")

all imports loaded — PIL, torch, datasets, unsloth, trl ready


In [ ]:
# Lazy image loading: keep only lightweight text rows in RAM; decode each image
# on demand when the trainer pulls it. This avoids holding 11k decoded images at once.
def load_image(name):
    img = Image.open(os.path.join(IMG_DIR, name)).convert("RGB")
    m = CFG["MAX_IMAGE_PX"]
    if max(img.size) > m:
        s = m / max(img.size)
        img = img.resize((int(img.size[0]*s), int(img.size[1]*s)))
    return img

def to_conversation(row):
    return {"messages": [
        {"role":"user","content":[
            {"type":"image","image": load_image(row["image"])},
            {"type":"text","text": INSTRUCTION + "\n\nQuestion: " + row["question"]}]},
        {"role":"assistant","content":[{"type":"text","text": row["target_output"]}]},
    ]}

class LazyConvDataset:
    """Builds each conversation (and loads its image) only when accessed."""
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        if isinstance(i, slice):
            return LazyConvDataset(self.rows[i])   # slicing -> a smaller lazy dataset
        return to_conversation(self.rows[i])
    def select(self, idxs):                        # HF-style helper, used by some trainers
        return LazyConvDataset([self.rows[k] for k in idxs])

train_conv = LazyConvDataset(train_rows)
print("lazy dataset ready:", len(train_conv), "samples (images load on demand)")
print("sample user text:\n", train_conv[0]["messages"][0]["content"][1]["text"][:200])

# def load_image(name):
#     img = Image.open(os.path.join(IMG_DIR, name)).convert("RGB")
#     m = CFG["MAX_IMAGE_PX"]
#     if max(img.size) > m:
#         s = m / max(img.size); img = img.resize((int(img.size[0]*s), int(img.size[1]*s)))
#     return img

# def to_conversation(row):
#     return {"messages": [
#         {"role":"user","content":[
#             {"type":"image","image": load_image(row["image"])},
#             {"type":"text","text": INSTRUCTION + "\n\nQuestion: " + row["question"]}]},
#         {"role":"assistant","content":[{"type":"text","text": row["target_output"]}]},
#     ]}

# train_conv = [to_conversation(r) for r in train_rows]
# print("formatted", len(train_conv), "samples")
# print("sample user text:\n", train_conv[0]["messages"][0]["content"][1]["text"][:200])

lazy dataset ready: 10450 samples (images load on demand)
sample user text:
 You are a circuit analysis tutor. Look at the schematic and answer the question. First list the components, then state the topology (what is in series/parallel), then write the equations and solve ste


## 5. Load model + attach LoRA

We finetune both vision and language layers: component *identification* is a vision-side skill, equation *setup* is language-side, and this task needs both.

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    CFG["MODEL"], load_in_4bit=True, use_gradient_checkpointing="unsloth")
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=CFG["LORA_R"], lora_alpha=CFG["LORA_ALPHA"], lora_dropout=0,
    bias="none", random_state=CFG["SEED"])
print("model + LoRA ready")

Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

==((====))==  Unsloth 2026.7.1: Fast Qwen2_5_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

model + LoRA ready


## 6. Day-1 OOM smoke test — the go/no-go gate

**Run this before the full training.** It trains 5 steps on a handful of samples and reports peak VRAM. Purpose: find out *today* whether the model trains without OOM at your image resolution, instead of discovering it 3 days in.

- **Passes** (completes, peak VRAM leaves headroom) → proceed to full training.
- **OOMs** → in order: (1) drop `MAX_IMAGE_PX` to 512 and re-run cell 4; (2) set `finetune_vision_layers=False`; (3) switch `CFG["MODEL"]` to a 2B (e.g. SmolVLM) and reload cell 5. Re-run this gate until it passes.


In [ ]:
import time
FastVisionModel.for_training(model)
smoke = SFTTrainer(
    model=model, tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=LazyConvDataset(train_rows[:8]),
    args=SFTConfig(
        per_device_train_batch_size=CFG["BATCH"], gradient_accumulation_steps=1,
        warmup_steps=0, max_steps=5, learning_rate=CFG["LR"], logging_steps=1,
        optim="adamw_8bit", weight_decay=0.001, lr_scheduler_type="linear",
        seed=CFG["SEED"], output_dir="smoke_out", report_to="none",
        remove_unused_columns=False, dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True}, max_length=CFG["MAX_LEN"],
    ))
torch.cuda.reset_peak_memory_stats()
t=time.time(); smoke.train(); dt=time.time()-t
peak = torch.cuda.max_memory_reserved()/1e9
total = torch.cuda.get_device_properties(0).total_memory/1e9
print(f"\nSMOKE TEST PASSED: 5 steps in {dt:.0f}s | peak VRAM {peak:.1f} / {total:.1f} GB")
print("GO." if peak < 0.9*total else "TIGHT - lower MAX_IMAGE_PX before the full run.")

Unsloth: Model does not have a default image size - using 512


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8 | Num Epochs = 1 | Total steps = 5
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 1 x 1) = 1
 "-____-"     Trainable parameters = 41,084,928 of 3,795,707,904 (1.08% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.722866
2,2.113713
3,2.779430
4,2.187795
5,1.609380


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Unsloth: Restored added_tokens_decoder metadata in smoke_out/checkpoint-5/tokenizer_config.json.



SMOKE TEST PASSED: 5 steps in 91s | peak VRAM 5.7 / 15.6 GB
GO.


## 7. Full training

**This is capped by `CFG["MAX_STEPS"]`, not epochs.** A full epoch over ~10k images at effective batch 4 is ~2,600 optimizer steps (≈6 h on a T4) — that's *steps*, not epochs. This behavior is highly structured and narrow, so a few hundred steps is plenty. `MAX_STEPS=150` at `MAX_IMAGE_PX=512` runs in roughly **15–25 min on a T4**. The trainer shuffles the full dataset, so 150 steps still samples across all three families.

**Scale strategy:** run 150 steps, look at the section-9 eval curve, and only raise `MAX_STEPS` (or set it to `None` for a full epoch) if accuracy is still climbing — extra steps past diminishing returns just burn time. If you OOM or it's too slow, keep `MAX_IMAGE_PX=512`; if you have headroom and want sharper boxes, try 640/768.

In [ ]:
FastVisionModel.for_training(model)
eff_batch = CFG["BATCH"]*CFG["GRAD_ACCUM"]
use_steps = CFG.get("MAX_STEPS")
schedule  = dict(max_steps=use_steps) if use_steps else dict(num_train_epochs=CFG["EPOCHS"])
seen      = use_steps*eff_batch if use_steps else len(train_conv)
print(f"training: {'max_steps='+str(use_steps) if use_steps else 'epochs='+str(CFG['EPOCHS'])}"
      f" | effective batch {eff_batch} | ~{seen} images seen (dataset has {len(train_conv)}, shuffled)")

# periodic checkpoints so a disconnect does not lose the run (keeps the 2 most recent)
if CFG.get("SAVE_TO_DRIVE"):
    from google.colab import drive; drive.mount("/content/drive")
    CKPT_DIR = os.path.join(CFG["DRIVE_DIR"], "checkpoints")   # written straight to Drive during training
else:
    CKPT_DIR = CFG["OUT_DIR"]                                    # /content/circuitsight_qlora (download manually)
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"checkpoints -> {CKPT_DIR}  (every 100 steps, keeping the 2 most recent)")

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_conv,
    args=SFTConfig(
        per_device_train_batch_size=CFG["BATCH"], gradient_accumulation_steps=CFG["GRAD_ACCUM"],
        warmup_steps=5, learning_rate=CFG["LR"],
        logging_steps=10, optim="adamw_8bit", weight_decay=0.001, lr_scheduler_type="linear",
        seed=CFG["SEED"], output_dir=CKPT_DIR, report_to="none",
        remove_unused_columns=False, dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True}, max_length=CFG["MAX_LEN"],
        save_strategy="steps", save_steps=100, save_total_limit=2,   # mid-run checkpoints for disconnect safety (keep 2 newest)
        **schedule,
    ))
stats = trainer.train()
print("done. final loss:", round(stats.training_loss, 4))

Unsloth: Model does not have a default image size - using 512


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,450 | Num Epochs = 1 | Total steps = 2,613
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 41,084,928 of 3,795,707,904 (1.08% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
25,1.009685
50,0.126995
75,0.058548
100,0.052084
125,0.046744
150,0.038322
175,0.040585
200,0.038712
225,0.041156
250,0.039225


Unsloth: Restored added_tokens_decoder metadata in circuitsight_qlora/checkpoint-500/tokenizer_config.json.


## 8. Inference helper

In [ ]:
def solve_image(pil_img, question, model, tokenizer, max_new_tokens=400):
    FastVisionModel.for_inference(model)
    msgs = [{"role":"user","content":[{"type":"image"},
             {"type":"text","text": INSTRUCTION + "\n\nQuestion: " + question}]}]
    text = tokenizer.apply_chat_template(msgs, add_generation_prompt=True)
    inputs = tokenizer(pil_img, text, add_special_tokens=False, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True,
                         do_sample=False, temperature=0.0)
    return tokenizer.decode(out[0], skip_special_tokens=True).split("assistant")[-1].strip()

# quick look
r0 = val_rows[0] if val_rows else train_rows[0]
print("Q:", r0["question"])
print(solve_image(load_image(r0["image"]), r0["question"], model, tokenizer)[:500])

## 9. Evaluation harness (base vs. tuned)

This is the point of the whole project. The harness parses each model output (preferring the machine-readable `FINAL: {quantity, target_id, value, unit, abstain}` line, with a prose fallback) and scores it against the exact gold labels along separate axes:

- **component_accuracy** — right resistor count *and* the right component **types** present (R vs C vs L).
- **reactive_type_accuracy** — on capacitor/inductor circuits, did it identify the reactive component as the correct type (not confuse a capacitor for an inductor)? This is the headline **"spatial blindness"** number.
- **Req_accuracy / answer_accuracy / step_verified_accuracy** — did it get the final quantity right, *and* the `R_eq` intermediate right (a correct answer from a wrong intermediate does **not** count).
- **grounding_accuracy** (IoU ≥ 0.5 vs. gold boxes) and **grounding_hallucination_rate** — did the cited regions actually land on the components, or invent locations?
- **fabrication_rate** vs. **honest_abstention_rate** — on illegible-value cases, did it invent a number (bad) or correctly say null (good)?
- **concept_hallucination_rate** — did it invoke a solving concept the problem doesn't need?
- **answer_accuracy_by_qtype** — a breakdown across current / voltage / power / resistance / charge / energy.

We run it on the **base** model and the **tuned** model over the same held-out set and compare. Report the **real-world** transfer set (§10) as the headline number.


In [ ]:
import re, json, random
import sympy as smp
# ============================================================================
# Eval harness for the generalized schema:
#   FINAL: {quantity, target_id, value, unit, abstain}
# Scores, separately, the axes the BrainLift cares about:
#   component identification (count + TYPE: R vs C vs L), grounding (IoU>=0.5) +
#   grounding-hallucination, step-verified solve (final AND R_eq intermediate),
#   fabrication vs honest-abstention, concept-hallucination, and — the headline
#   "spatial blindness" number — reactive component-type accuracy (cap vs inductor).
# ============================================================================
def parse_final(text):
    m = re.search(r"FINAL:\s*(\{.*\})", text, re.DOTALL)
    if m:
        try:
            j = json.loads(m.group(1))
            return {"quantity":j.get("quantity"),"target_id":j.get("target_id"),
                    "value":j.get("value"),"unit":j.get("unit"),"abstain":bool(j.get("abstain",False))}
        except Exception: pass
    low = text.lower()                                     # prose fallback
    ab = ("null" in low) or ("not legible" in low) or ("cannot" in low)
    val=None
    m2 = re.search(r"answer[^=]*=\s*[^=]*?(-?\d+(?:\.\d+)?(?:e-?\d+)?)", low)
    if m2:
        try: val=float(m2.group(1))
        except Exception: val=None
    return {"quantity":None,"target_id":None,"value":val,"unit":None,"abstain":ab}

def parse_components(text):
    res_ids = set(re.findall(r"\bR(\d+)\b", text))
    has_cap = bool(re.search(r"capacitor|\u00b5F|uF|\bC1\b", text, re.I))
    has_ind = bool(re.search(r"inductor|\bmH\b|\bL1\b", text, re.I))
    m = re.search(r"(\d+)\s*resistor", text.lower())
    return {"n_res": int(m.group(1)) if m else len(res_ids), "has_cap":has_cap, "has_ind":has_ind}

def parse_boxes(text):
    pat=r"(V\d+|R\w+|C\w+|L\w+|SW|VM)\s*<box>\s*\[?\s*(-?\d+)\s*,\s*(-?\d+)\s*,\s*(-?\d+)\s*,\s*(-?\d+)\s*\]?\s*</box>"
    return {cid:[int(a),int(b),int(c),int(d)] for cid,a,b,c,d in re.findall(pat, text)}

def _iou(p,g):
    ix0,iy0,ix1,iy1=max(p[0],g[0]),max(p[1],g[1]),min(p[2],g[2]),min(p[3],g[3])
    inter=max(0,ix1-ix0)*max(0,iy1-iy0)
    u=max(0,p[2]-p[0])*max(0,p[3]-p[1])+max(0,g[2]-g[0])*max(0,g[3]-g[1])-inter
    return inter/u if u>0 else 0.0

_CKEYS={"ohm":"ohm","parallel":"parallel","series":"series","capacitor":"capacitor","inductor":"inductor"}
def _concepts_text(text):
    m=re.search(r"concepts used:\s*(.+)", text.lower())
    if not m: return None
    chunk=m.group(1).split("\n")[0]
    return {v for k,v in _CKEYS.items() if k in chunk}
def _concepts_gold(concepts):
    s=" ".join(concepts).lower(); return {v for k,v in _CKEYS.items() if k in s}

_NUM=re.compile(r"-?\d+\.?\d*(?:[eE][+-]?\d+)?$")
_SYM_RNG=random.Random(12345)   # fixed seed -> reproducible symbolic-equivalence checks
def _is_num(x):
    return isinstance(x,(int,float)) or (isinstance(x,str) and bool(_NUM.match(x.strip())))
def sym_equal(a, b, trials=6):
    """Algebraic equivalence via random substitution: plug the same random reals into the
    shared free symbols and compare (robust to any equivalent form, e.g. R1+R2 == R2+R1)."""
    try:
        ea=smp.sympify(str(a)); eb=smp.sympify(str(b))
    except Exception:
        return str(a).replace(" ","")==str(b).replace(" ","")
    syms=sorted(ea.free_symbols | eb.free_symbols, key=str)
    if not syms:
        try: return abs(float(ea)-float(eb))<=1e-6*max(1.0,abs(float(eb)))
        except Exception:
            try: return smp.simplify(ea-eb)==0
            except Exception: return False
    for _ in range(trials):
        subsd={s:_SYM_RNG.uniform(1.5,9.5) for s in syms}
        try: fa=float(ea.subs(subsd)); fb=float(eb.subs(subsd))
        except Exception:
            try: return smp.simplify(ea-eb)==0
            except Exception: return False
        if abs(fa-fb) > 1e-6*max(1.0,abs(fb)): return False
    return True
def _close(pred, gold, rel=0.03):
    """Numeric (3% tol) when both numeric; else algebraic equivalence (symbolic answers)."""
    if pred is None or gold is None: return False
    if _is_num(pred) and _is_num(gold):
        return abs(float(pred)-float(gold)) <= max(1e-9, abs(float(gold))*rel)
    return sym_equal(pred, gold)

def score_record(gold, model_text, rel_tol=0.03, ground_mode="id"):
    ga=bool(gold["abstain"]); fam=gold.get("family","dc_resistor")
    p=parse_final(model_text); comp=parse_components(model_text); mb=parse_boxes(model_text)
    gc=gold["gold_components"]
    res={"family":fam,"question_type":gold.get("question_type"),
         "abstain_gold":ga,"abstain_pred":bool(p["abstain"]),
         "comp_ok":None,"react_type_ok":None,"req_ok":None,"answer_ok":None,"intermediates_ok":None,
         "fabricated":None,"honest_abstain":None,
         "concept_declared":None,"concept_hallucinated":None,
         "grounding_ok":None,"grounding_hallucinated":None}
    # component identification: right resistor COUNT and right presence of cap/inductor TYPE
    res["comp_ok"]=(comp["n_res"]==gc.get("resistor")) and \
                   (comp["has_cap"]==("capacitor" in gc)) and (comp["has_ind"]==("inductor" in gc))
    if fam=="reactive":                                    # cap-vs-inductor confusion (headline)
        want_cap="capacitor" in gc
        res["react_type_ok"]=bool((comp["has_cap"] and not comp["has_ind"]) if want_cap
                                  else (comp["has_ind"] and not comp["has_cap"]))
    # grounding: fraction of gold components boxed with IoU>=0.5; hallucination = boxes that miss
    gb=gold.get("gold_boxes",{})   # ground_mode="iou": id-agnostic greedy match (real images number their own way)
    if gb:
        if ground_mode=="iou":
            preds=list(mb.values()); used=[False]*len(preds); hit=0
            for g in gb.values():
                best=0.0; bi=-1
                for i,pbx in enumerate(preds):
                    if used[i]: continue
                    v=_iou(pbx,g)
                    if v>best: best=v; bi=i
                if bi>=0 and best>=0.5: used[bi]=True; hit+=1
            res["grounding_ok"]=hit/len(gb)
            if preds: res["grounding_hallucinated"]=sum(1 for u in used if not u)/len(preds)
        else:
            res["grounding_ok"]=sum(1 for cid,g in gb.items() if cid in mb and _iou(mb[cid],g)>=0.5)/len(gb)
            if mb:
                res["grounding_hallucinated"]=sum(1 for cid,m in mb.items()
                                                  if cid not in gb or _iou(m,gb[cid])<0.5)/len(mb)
    # concept scoping
    gset=_concepts_gold(gold.get("concepts",[])); pc=_concepts_text(model_text)
    if pc is None: res["concept_declared"]=False
    else: res["concept_declared"]=True; res["concept_hallucinated"]=len(pc-gset)>0
    # answer / abstention
    if ga:
        gave=(p["value"] is not None) and (not p["abstain"])
        res["fabricated"]=gave; res["honest_abstain"]=bool(p["abstain"]) and not gave
    else:
        ga_dict=gold.get("gold_answer") or {}          # perception-only records have gold_answer=None
        gv=ga_dict.get("value")                        # float | expr-string | None
        gvals=gold.get("gold_values") or {}
        res["answer_ok"]=_close(p["value"], gv, rel_tol) if gv is not None else None
        qt=gold.get("question_type")
        if qt=="resistance":
            res["req_ok"]=res["answer_ok"]
        elif qt in ("topology","components") or gv is None:   # perception-only: no numeric intermediate
            res["req_ok"]=None
        elif gvals.get("I_total")==0:                  # open circuit: no loop-R_eq intermediate
            res["req_ok"]=None
        elif "R_eq" in gvals:
            gold_req=gvals["R_eq"]
            if _is_num(gold_req):
                mR=re.search(r"r_eq\s*=\s*(-?\d+\.?\d*(?:[eE][+-]?\d+)?)", model_text, re.I)
                predR=mR.group(1) if mR else None
            else:
                mR=re.search(r"r_eq\s*=\s*([^\n]+?)\s*(?:ohm|\u03a9|\(=|$)", model_text, re.I) or \
                   re.search(r"r_eq\s*=\s*([^\n.]+)", model_text, re.I)
                predR=mR.group(1).strip().rstrip(".") if mR else None
            res["req_ok"]=_close(predR, gold_req, rel_tol)
        gbc=gvals.get("branch_currents",{})            # full intermediate verification
        if gbc:
            pbc={cid:v.strip() for cid,v in re.findall(r"I\((R\w+)\)\s*=\s*([^,\n]+?)\s*A", model_text)}
            checked=ok=0
            for cid,gvv in gbc.items():
                if _is_num(gvv) and abs(float(gvv))<1e-3: continue   # skip sub-mA (numeric rounding noise)
                checked+=1; pv=pbc.get(cid)
                if pv is not None and _close(pv, gvv, rel_tol): ok+=1
            res["intermediates_ok"]=(ok==checked) if checked else None
    return res

def aggregate(results):
    non=[r for r in results if not r["abstain_gold"]]; ab=[r for r in results if r["abstain_gold"]]
    rj =[r for r in results if r["family"]=="reactive"]
    def frac(xs,k):
        xs=[r[k] for r in xs if r[k] is not None]; return round(sum(xs)/len(xs),4) if xs else None
    step=[r for r in non if r["comp_ok"] and (r["req_ok"] in (True,None)) and (r["intermediates_ok"] in (True,None)) and r["answer_ok"]]
    qts=sorted(set(r["question_type"] for r in non if r["question_type"]))
    _gr=frac(results,"grounding_ok"); _gh=frac(results,"grounding_hallucinated")
    _gp=(round(1-_gh,4) if _gh is not None else None)
    _gf1=(round(2*_gp*_gr/(_gp+_gr),4) if (_gp and _gr and _gp+_gr>0) else None)
    _tp=sum(1 for r in results if r["abstain_gold"] and r["abstain_pred"])
    _fp=sum(1 for r in results if (not r["abstain_gold"]) and r["abstain_pred"])
    _fn=sum(1 for r in results if r["abstain_gold"] and not r["abstain_pred"])
    _apr=(round(_tp/(_tp+_fp),4) if _tp+_fp else None); _arc=(round(_tp/(_tp+_fn),4) if _tp+_fn else None)
    return {"n_total":len(results),"n_abstain":len(ab),"n_reactive":len(rj),
            "component_accuracy":frac(results,"comp_ok"),
            "reactive_type_accuracy":frac(rj,"react_type_ok"),
            "Req_accuracy":frac(non,"req_ok"),"answer_accuracy":frac(non,"answer_ok"),
            "intermediates_accuracy":frac(non,"intermediates_ok"),
            "step_verified_accuracy":round(len(step)/len(non),4) if non else None,
            "fabrication_rate":frac(ab,"fabricated"),"honest_abstention_rate":frac(ab,"honest_abstain"),
            "grounding_accuracy":frac(results,"grounding_ok"),
            "grounding_hallucination_rate":frac(results,"grounding_hallucinated"),
            "concept_declared_rate":frac(results,"concept_declared"),
            "concept_hallucination_rate":frac(results,"concept_hallucinated"),
            "grounding_recall":_gr,"grounding_precision":_gp,"grounding_f1":_gf1,
            "abstention_precision":_apr,"abstention_recall":_arc,
            "answer_accuracy_by_qtype":{q:frac([r for r in non if r["question_type"]==q],"answer_ok") for q in qts}}

def evaluate(model, tokenizer, rows, n=None, ground_mode="id"):
    # ground_mode="iou" for the real-world set (model numbers components its own way)
    rows = rows[:n] if n else rows
    return aggregate([score_record(r, solve_image(load_image(r["image"]), r["question"], model, tokenizer),
                                    ground_mode=ground_mode)
                      for r in rows])
print("eval harness loaded (generalized schema + grounding + component-type confusion)")

In [ ]:
# Base vs tuned on the held-out synthetic val set (use the real-world eval set for the headline number).
EVAL_ROWS = val_rows if val_rows else train_rows[-40:]
EVAL_N = min(60, len(EVAL_ROWS))

# --- tuned (current, fine-tuned model) ---
tuned_metrics = evaluate(model, tokenizer, EVAL_ROWS, EVAL_N)
print("TUNED :", tuned_metrics)

# --- base (reload a fresh, un-tuned model) ---
base_model, base_tok = FastVisionModel.from_pretrained(
    CFG["MODEL"], load_in_4bit=True, use_gradient_checkpointing="unsloth")
FastVisionModel.for_inference(base_model)
base_metrics = evaluate(base_model, base_tok, EVAL_ROWS, EVAL_N)
print("BASE  :", base_metrics)

print("\n=== DELTA (tuned - base) ===")
for k in tuned_metrics:
    if isinstance(tuned_metrics[k],(int,float)) and isinstance(base_metrics.get(k),(int,float)):
        print(f"{k:28s}: {base_metrics[k]:.3f} -> {tuned_metrics[k]:.3f}  ({tuned_metrics[k]-base_metrics[k]:+.3f})")

# per-question-type answer accuracy (where the tuned model helps most)
print("\n--- answer accuracy by question type (base -> tuned) ---")
bt = base_metrics.get("answer_accuracy_by_qtype",{}); tt = tuned_metrics.get("answer_accuracy_by_qtype",{})
for q in sorted(set(bt)|set(tt)):
    b,t = bt.get(q), tt.get(q)
    bs = "  n/a" if b is None else f"{b:.3f}"; ts = "  n/a" if t is None else f"{t:.3f}"
    print(f"  {q:12s}: {bs} -> {ts}")

## 10. Real-world evaluation (the headline number)

Synthetic accuracy overstates real performance. Fill in the hand-labeled real set (`real_eval_TEMPLATE.json` from the data-gen notebook), load it the same way, and run `evaluate(...)` on it. Report **base vs. tuned on the real set** as your main result, and report the synthetic→real gap honestly.

In [ ]:
# Real-world transfer eval (the headline number). Upload the hand-labeled set to REAL_DIR — the
# whole data/real_eval/ folder, keeping its subpaths (labeled.jsonl + figures/... + opensource/...).
REAL_DIR  = "/content/real_eval"
real_path = os.path.join(REAL_DIR, "labeled.jsonl")

def _real_img(name):                                  # images live under REAL_DIR (not the train IMG_DIR)
    img = Image.open(os.path.join(REAL_DIR, name)).convert("RGB")
    m = CFG["MAX_IMAGE_PX"]
    if max(img.size) > m:
        s = m/max(img.size); img = img.resize((int(img.size[0]*s), int(img.size[1]*s)))
    return img

def eval_real(mdl, tok):
    # ground_mode="iou": match predicted->gold boxes by IoU regardless of id (the model numbers
    # real components its own way, so "R1" vs gold "Rx" must not fail grounding).
    res=[score_record(r, solve_image(_real_img(r["image"]), r["question"], mdl, tok), ground_mode="iou")
         for r in real_rows]
    return aggregate(res)

if os.path.exists(real_path):
    real_rows=[json.loads(l) for l in open(real_path)]
    for r in real_rows: r.setdefault("abstain", False)
    print(f"real records: {len(real_rows)}")
    print("REAL tuned:", eval_real(model, tokenizer))
    print("REAL base :", eval_real(base_model, base_tok))
    # NOTE: 6/8 real records are symbolic (perception-only) -> read component_accuracy,
    # reactive_type_accuracy, grounding_accuracy; answer/step-verified apply to the numeric ones.
else:
    print(f"No {real_path} yet - upload data/real_eval/ there (labeled.jsonl + figures/ + opensource/).")

## 11. Save / push the adapter

In [ ]:
model.save_pretrained(CFG["OUT_DIR"]); tokenizer.save_pretrained(CFG["OUT_DIR"])
print("saved LoRA adapter to", CFG["OUT_DIR"])

if CFG.get("SAVE_TO_DRIVE"):
    import os, shutil, time, glob
    from google.colab import drive; drive.mount("/content/drive")
    models_dir = os.path.join(CFG["DRIVE_DIR"], "models"); os.makedirs(models_dir, exist_ok=True)
    stamp = time.strftime("%Y%m%d_%H%M%S")
    snap = shutil.make_archive(os.path.join(models_dir, f"circuitsight_qlora_{stamp}"), "zip", CFG["OUT_DIR"])
    print("model snapshot saved to Drive:", snap)
    # keep only the 2 most recent snapshots so Drive does not fill up
    snaps = sorted(glob.glob(os.path.join(models_dir, "circuitsight_qlora_*.zip")), key=os.path.getmtime)
    for old in snaps[:-2]:
        os.remove(old); print("  pruned old snapshot:", os.path.basename(old))
    print("  kept:", [os.path.basename(s) for s in snaps[-2:]])
# To the Hub:
# from huggingface_hub import login; login()
# model.push_to_hub("your-username/circuitsight-qwen2.5vl-3b")
# tokenizer.push_to_hub("your-username/circuitsight-qwen2.5vl-3b")
# Merged 16-bit for deployment:
# model.save_pretrained_merged("circuitsight_merged", tokenizer)

## 12. For graders — load the fine-tuned model and run it on one image

No HuggingFace account needed. In a **fresh runtime**, the minimum path is:
1. Run **§1** (install) → **Restart session** → the **imports** cell → the **§2 Config** cell (defines `CFG` + `INSTRUCTION`).
2. Get the adapter: unzip the submitted `circuitsight_qlora_*.zip` and point `ADAPTER_DIR` at the unzipped folder (it contains `adapter_config.json`).
3. Set `IMAGE_PATH` to a circuit image (upload one via the file browser) and run the cell below.

Right after training (same session) it just reuses the model already in memory.

In [ ]:
# ---- FOR GRADERS: reload the fine-tuned model and run it on ONE circuit image ----
ADAPTER_DIR = CFG["OUT_DIR"]   # folder with adapter_config.json; or an unzipped circuitsight_qlora_*.zip snapshot
IMAGE_PATH  = ""               # <- set to your circuit image, e.g. "/content/my_circuit.png"
QUESTION    = "Identify the components (with their image regions), state the topology, and solve the circuit."

from unsloth import FastVisionModel
from PIL import Image

if "model" in globals() and "tokenizer" in globals():
    _m, _t = model, tokenizer                      # reuse the model from this training session (no extra VRAM)
    print("using the in-session fine-tuned model")
else:                                              # fresh runtime: load base (4-bit) + our LoRA adapter
    _m, _t = FastVisionModel.from_pretrained(ADAPTER_DIR, load_in_4bit=True,
                                             use_gradient_checkpointing="unsloth")
    print("loaded fine-tuned model from", ADAPTER_DIR)
    # fallback if your Unsloth version won't load an adapter dir directly:
    #   from peft import PeftModel
    #   _m, _t = FastVisionModel.from_pretrained(CFG["MODEL"], load_in_4bit=True)
    #   _m = PeftModel.from_pretrained(_m, ADAPTER_DIR)
FastVisionModel.for_inference(_m)

def run_circuit(image_path, question=QUESTION, max_new_tokens=400):
    img = Image.open(image_path).convert("RGB")
    m = CFG["MAX_IMAGE_PX"]
    if max(img.size) > m:
        s = m/max(img.size); img = img.resize((int(img.size[0]*s), int(img.size[1]*s)))
    msgs=[{"role":"user","content":[{"type":"image"},
           {"type":"text","text": INSTRUCTION + "\n\nQuestion: " + question}]}]
    text=_t.apply_chat_template(msgs, add_generation_prompt=True)
    inputs=_t(img, text, add_special_tokens=False, return_tensors="pt").to("cuda")
    out=_m.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=0.0, use_cache=True)
    return _t.decode(out[0], skip_special_tokens=True).split("assistant")[-1].strip()

if IMAGE_PATH:
    print("\nQ:", QUESTION, "\n")
    print(run_circuit(IMAGE_PATH))
else:
    print("\nSet IMAGE_PATH to a circuit image (upload one via the file browser) and re-run this cell.")

## 12. Notes & troubleshooting

- **`could not make a flat list of images`** → ensure `to_conversation` passes PIL images (it does) and exactly one image per sample.
- **`max_length` vs `max_seq_length`** → newer TRL uses `max_length` (used here). If your TRL errors, rename it to `max_seq_length`.
- **OOM mid-run** → lower `MAX_IMAGE_PX` (768→512), keep `use_gradient_checkpointing="unsloth"`, `BATCH=1`, raise `GRAD_ACCUM`; last resort, `finetune_vision_layers=False` or a 2B model.
- **Tuned barely beats base** → concentrate the eval on harder cases (more components, parallel blocks) and on the abstention subset, where the base fails most; check the loss actually dropped; add render variety in the data.
- **Scaling** → build data for 50k but train up from ~10-15k, watching the section-9 curve; stop when it flattens.


## 13. (v2) DPO preference tuning — build this AFTER v1 succeeds

> **Do not build/run this yet.** DPO is a *second stage that runs on top of the v1 fine-tuned model*, not an alternative to it. It only becomes meaningful — and measurable — once v1 has posted a solid base-vs-tuned gain. Running it on a weak v1 model just adds noise you can't interpret.

**When to come back here:** after section 9 shows the tuned model clearly beating the base on component / R_eq / answer accuracy (and ideally on the real-world set in section 10).

**What it will do when built:**
- Load `train_pairs.jsonl` (already produced by the data notebook when `INCLUDE_MISTAKES=True`).
- Form preference pairs: `correct_solution` = *chosen*, `wrong_solution` = *rejected*, same image + question as the prompt.
- Run DPO on top of the v1 LoRA adapter (Unsloth supports vision DPO), which sharpens the model *away* from the exact mistakes in the pairs (parallel-as-series, omitted branch, Ohm's-law flip, misread value).
- Re-run the section-9 harness to check DPO improved spec adherence *beyond* SFT alone — especially the fabrication / setup-error cases.

**Why it's deferred, not written now:** vision DPO has its own trainer, a reference-model copy (tighter VRAM than SFT), and settings that depend on what v1 actually produced and which model/hardware the Day-1 smoke test landed on. Writing it before v1 exists risks writing it twice. The data is already waiting, so nothing is blocked by deferring.